In [1]:
from src.data_collection.osm_extractor import extract_osm_pois
from src.data_collection.data_cleaning.cooking_column import classify_cuisine_column
from src.data_collection.bbox_reshap import bbox_wgs84_to_lambert
from src.config import *
import numpy as np

In [9]:
df_osm = extract_osm_pois(BBOX, CATEGORIES, use_cache=False)
df_osm = classify_cuisine_column(df_osm)
poi_all_index = df_osm.index.tolist()
df_osm["index"] = poi_all_index
df_osm = df_osm[df_osm['geometry'].within(bbox_wgs84_to_lambert(BBOX))]
df_osm = df_osm[["index", "name", "category", "geometry", "lat", "lon", "x", "y", "indoor_seating", "outdoor_seating"]]
df_osm = df_osm.dropna(subset=["name"])
df_osm = df_osm.rename(columns={"index": "poi_id", "name": "poi_name"})

/Users/sese/Documents/code/benchmark_pipeline/src/data_collection/osm_extractor.py:130: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


In [32]:
df_osm["category"].unique()

array(['station', 'restaurant', 'hotel', 'bar', 'cafe', 'subway_entrance',
       'pub', 'garden', 'park', 'grass', 'nature_reserve'], dtype=object)

In [ ]:
def sample_anchors(df, nb_q=110, seed=42):
    rng = np.random.default_rng(seed)
    list_cat = df_osm["category"].unique()
    n_queries_per_stratum = nb_q // len(list_cat)
    for cat_anc in list_cat:
        sub = df_osm[df_osm['category'] == cat_anc]
        anchors = sub.sample(n=min(n_queries_per_stratum, len(sub)), random_state=rng)
    return anchors

In [ ]:
from collections import defaultdict
import duckdb
import pandas as pd

def near_sql(df, x, y, cat, k=100):
    con = duckdb.connect()
    con.sql("INSTALL spatial; LOAD spatial;")
    df = pd.DataFrame(df.drop(columns="geometry")).assign(geom_wkb=df.geometry.to_wkb())
    results = con.execute("""
        SELECT poi_id, poi_name,
                ST_Distance(ST_GeomFromWKB(geom_wkb), ST_Point($x, $y))::DOUBLE AS dist
        FROM df
        WHERE category = $cat
        ORDER BY dist
        LIMIT $k
    """, {"x": x, "y": y, "cat": cat, "k": k}).df()
    return results

def make_question_nearsql(df_osm, nb_q=110, seed=42):
    dic_benchmark = defaultdict(list)
    rng = np.random.default_rng(seed)
    list_cat = df_osm["category"].unique()
    n_queries_per_stratum = nb_q // len(list_cat)
    for cat_anc in list_cat:
        sub = df_osm[df_osm['category'] == cat_anc]
        anchors = sub.sample(n=min(n_queries_per_stratum, len(sub)), random_state=rng)
        for anchor in anchors.itertuples():
            cat_q = rng.choice(list_cat)
            results = near_sql(df_osm, anchor.x, anchor.y, cat_q)
            if cat_anc == cat_q:
                results = results[results["poi_id"] != anchor.poi_id]

            dic_benchmark["query"].append(f"{cat_q} near {anchor.poi_name}")
            dic_benchmark["anchor_index"].append(anchor.poi_id)
            dic_benchmark["anchor_name"].append(anchor.poi_name)
            dic_benchmark["anchor_category"].append(anchor.category)
            dic_benchmark["anchor_x"].append(anchor.x)
            dic_benchmark["anchor_y"].append(anchor.y)
            dic_benchmark["category_query"].append(cat_q)
            dic_benchmark["results"].append(results)
            dic_benchmark["same_cat"].append(cat_anc==cat_q)
            dic_benchmark["function"].append("near_sql")
    return pd.DataFrame(dic_benchmark)

bench = make_question_nearsql(df_osm)

    

In [ ]:
list_distance = [100, 300, 500, 1000]

def near_metric_sql(df, x, y, cat, distance, k=100):
    con = duckdb.connect()
    con.sql("INSTALL spatial; LOAD spatial;")
    df = pd.DataFrame(df.drop(columns="geometry")).assign(geom_wkb=df.geometry.to_wkb())

    results = con.execute("""
        SELECT poi_id, poi_name,
                ST_Distance(ST_GeomFromWKB(geom_wkb), ST_Point($x, $y))::DOUBLE AS dist
        FROM df
        WHERE category = $cat AND dist < $d
        ORDER BY dist
        LIMIT $k
    """, {"x": x, "y": y, "cat": cat, "k": k, "d": distance}).df()

    return results

def make_question_metricsql(df_osm, list_distance, nb_q=110, seed=42):
    dic_benchmark = defaultdict(list)
    rng = np.random.default_rng(seed)
    list_cat = df_osm["category"].unique()
    n_queries_per_stratum = nb_q // len(list_cat)
    for cat_anc in list_cat:
        sub = df_osm[df_osm['category'] == cat_anc]
        anchors = sub.sample(n=min(n_queries_per_stratum, len(sub)), random_state=rng)
        for anchor in anchors.itertuples():
            cat_q = rng.choice(list_cat)
            for d in list_distance:
                results = near_metric_sql(df_osm, anchor.x, anchor.y, cat_q, distance=d)
                if cat_anc == cat_q:
                    results = results[results["poi_id"] != anchor.poi_id]

                dic_benchmark["query"].append(f"{cat_q} at less than {d} meters from {anchor.poi_name}")
                dic_benchmark["anchor_index"].append(anchor.poi_id)
                dic_benchmark["anchor_name"].append(anchor.poi_name)
                dic_benchmark["anchor_category"].append(anchor.category)
                dic_benchmark["anchor_x"].append(anchor.x)
                dic_benchmark["anchor_y"].append(anchor.y)
                dic_benchmark["category_query"].append(cat_q)
                dic_benchmark["results"].append(results)
                dic_benchmark["same_cat"].append(cat_anc==cat_q)
                dic_benchmark["distance"].append(d)
                dic_benchmark["function"].append("near_metric_sql")
    return pd.DataFrame(dic_benchmark)

In [24]:
make_question_metricsql(df_osm, list_distance, nb_q=110, seed=42)

,query,anchor_index,anchor_name,anchor_category,anchor_x,anchor_y,category_query,results,same_cat,distance,function
0,cafe at 100 meters from Porte Maillot,22878,Porte Maillot,station,647333.427105,6.864446e+06,cafe,"Empty DataFrame Columns: [poi_id, poi_name, di...",False,100,near_metric_sql
1,cafe at 300 meters from Porte Maillot,22878,Porte Maillot,station,647333.427105,6.864446e+06,cafe,poi_id poi_name dist 0 21464 ...,False,300,near_metric_sql
2,cafe at 500 meters from Porte Maillot,22878,Porte Maillot,station,647333.427105,6.864446e+06,cafe,poi_id poi_name dist ...,False,500,near_metric_sql
3,cafe at 1000 meters from Porte Maillot,22878,Porte Maillot,station,647333.427105,6.864446e+06,cafe,poi_id ...,False,1000,near_metric_sql
4,subway_entrance at 100 meters from Poissonnière,10375,Poissonnière,station,652234.314202,6.864319e+06,subway_entrance,poi_id poi_name dist 0 ...,False,100,near_metric_sql
...,...,...,...,...,...,...,...,...,...,...,...
399,subway_entrance at 1000 meters from Pelouse Je...,19255,Pelouse Jean Seberg,grass,647082.856123,6.863761e+06,subway_entrance,poi_id poi_name ...,False,1000,near_metric_sql
400,park at 100 meters from Le Petit Labyrinthe,21034,Le Petit Labyrinthe,nature_reserve,652842.811594,6.860580e+06,park,"Empty DataFrame Columns: [poi_id, poi_name, di...",False,100,near_metric_sql
401,park at 300 meters from Le Petit Labyrinthe,21034,Le Petit Labyrinthe,nature_reserve,652842.811594,6.860580e+06,park,poi_id poi_name dist ...,False,300,near_metric_sql
402,park at 500 meters from Le Petit Labyrinthe,21034,Le Petit Labyrinthe,nature_reserve,652842.811594,6.860580e+06,park,poi_id poi_n...,False,500,near_metric_sql


In [30]:
CARDINAL_AZ = {
    "north": 0, "east": 90, 
    "south": 180, "west": 270,
}

def cardinal_azimuth_sql(df, x, y, cat, cardinal_dir, k=100,
                        half_width=70.0, con=None):
    center = CARDINAL_AZ[cardinal_dir]

    if con is None:
        con = duckdb.connect()
        con.sql("INSTALL spatial; LOAD spatial;")
    flat = pd.DataFrame(df.drop(columns="geometry")).assign(geom_wkb=df.geometry.to_wkb())
    con.register("poi", flat)

    return con.execute("""
        WITH g AS (
            SELECT poi_id, poi_name, ST_GeomFromWKB(geom_wkb) AS geom
            FROM poi WHERE category = $cat
        ), d AS (
            SELECT poi_id, poi_name,
                   ST_Distance_Sphere(geom, ST_Point($x, $y))::DOUBLE AS dist,
                   (degrees(atan2(
                        (ST_X(geom) - $x) * cos(radians(($y + ST_Y(geom)) / 2)),
                        (ST_Y(geom) - $y)
                    )) + 360) % 360 AS az
            FROM g
        )
        SELECT poi_id, poi_name, dist, az,
               row_number() OVER (ORDER BY dist) AS rank
        FROM d
        WHERE abs(((az - $center + 540)::DOUBLE % 360) - 180) <= $half
        ORDER BY dist
        LIMIT $k
    """, {"x": x, "y": y, "cat": cat, "k": k,
          "center": center, "half": half_width}).df()

def make_question_cardinalazi(df_osm, list_direction=["north", "east", "west", "south"], half_width=70.0, nb_q=110, seed=42):
    dic_benchmark = defaultdict(list)
    rng = np.random.default_rng(seed)
    list_cat = df_osm["category"].unique()
    n_queries_per_stratum = nb_q // len(list_cat)
    for cat_anc in list_cat:
        sub = df_osm[df_osm['category'] == cat_anc]
        anchors = sub.sample(n=min(n_queries_per_stratum, len(sub)), random_state=rng)
        for anchor in anchors.itertuples():
            cat_q = rng.choice(list_cat)
            for d in list_direction:
                results = cardinal_azimuth_sql(df_osm, anchor.x, anchor.y, cat_q, cardinal_dir=d, half_width=half_width)
                if cat_anc == cat_q:
                    results = results[results["poi_id"] != anchor.poi_id]

                dic_benchmark["query"].append(f"{cat_q} at {d} meters from {anchor.poi_name}")
                dic_benchmark["anchor_index"].append(anchor.poi_id)
                dic_benchmark["anchor_name"].append(anchor.poi_name)
                dic_benchmark["anchor_category"].append(anchor.category)
                dic_benchmark["anchor_x"].append(anchor.x)
                dic_benchmark["anchor_y"].append(anchor.y)
                dic_benchmark["category_query"].append(cat_q)
                dic_benchmark["results"].append(results)
                dic_benchmark["same_cat"].append(cat_anc==cat_q)
                dic_benchmark["direction"].append(d)
                dic_benchmark["function"].append("cardinal_azimuth_sql")
    return pd.DataFrame(dic_benchmark)

In [31]:
make_question_cardinalazi(df_osm, list_direction=["north", "east", "west", "south"], half_width=70.0, nb_q=110, seed=42)

,query,anchor_index,anchor_name,anchor_category,anchor_x,anchor_y,category_query,results,same_cat,direction,function
0,cafe at north meters from Porte Maillot,22878,Porte Maillot,station,647333.427105,6.864446e+06,cafe,poi_id poi_name dist...,False,north,cardinal_azimuth_sql
1,cafe at east meters from Porte Maillot,22878,Porte Maillot,station,647333.427105,6.864446e+06,cafe,poi_id poi_name dist ...,False,east,cardinal_azimuth_sql
2,cafe at west meters from Porte Maillot,22878,Porte Maillot,station,647333.427105,6.864446e+06,cafe,poi_id poi_na...,False,west,cardinal_azimuth_sql
3,cafe at south meters from Porte Maillot,22878,Porte Maillot,station,647333.427105,6.864446e+06,cafe,poi_id poi_na...,False,south,cardinal_azimuth_sql
4,subway_entrance at north meters from Poissonnière,10375,Poissonnière,station,652234.314202,6.864319e+06,subway_entrance,poi_id poi_name ...,False,north,cardinal_azimuth_sql
...,...,...,...,...,...,...,...,...,...,...,...
399,subway_entrance at south meters from Pelouse J...,19255,Pelouse Jean Seberg,grass,647082.856123,6.863761e+06,subway_entrance,poi_id poi_name ...,False,south,cardinal_azimuth_sql
400,park at north meters from Le Petit Labyrinthe,21034,Le Petit Labyrinthe,nature_reserve,652842.811594,6.860580e+06,park,poi_id poi_nam...,False,north,cardinal_azimuth_sql
401,park at east meters from Le Petit Labyrinthe,21034,Le Petit Labyrinthe,nature_reserve,652842.811594,6.860580e+06,park,poi_id ...,False,east,cardinal_azimuth_sql
402,park at west meters from Le Petit Labyrinthe,21034,Le Petit Labyrinthe,nature_reserve,652842.811594,6.860580e+06,park,poi_id ...,False,west,cardinal_azimuth_sql


In [ ]:
"a faire "
"changer le fait que le tableau des resultast reste dans le tableau  sous forme de tabmeau pour que ce soit utilisable en parquet"
"faire les commentaires"
"faire mieux la stratification"